# Representation And Motif Diagnostics

This notebook reuses a selected training run, runs three diagnostics experiments in sequence, and exports the results under a dedicated diagnostics folder for that training `run_id`.

In [ ]:
# Configuration - edit these values, then Run All

# Shared
TRAINING_RUN_ID = None  # None = use the latest exported training run; or set a specific run_id like 'run_20260407_153000'
DISCOVERY_DEVICE_MODE = 'auto'  # 'auto' = GPU encoder if available, 'cpu' = force CPU only
STAGE_PREPARED_SESSIONS_LOCALLY = True  # recommended in Colab; copies just the discovery/test .h5 files off Drive before long runs
NOTEBOOK_STEP_LOG_EVERY = 16  # print a notebook log line every N steps from the CLI wrapper

# Experiment 1 - baseline stimulus_change + geometry diagnostics
BASE_DECODE_TYPE = 'stimulus_change'
BASE_DISCOVERY_MAX_BATCHES = 128
BASE_DISCOVERY_TOP_K_CANDIDATES = 64
BASE_DISCOVERY_CANDIDATE_SESSION_BALANCE_FRACTION = 0.2
BASE_DISCOVERY_MIN_CANDIDATE_SCORE = 0.0
BASE_DISCOVERY_MIN_CLUSTER_SIZE = 2
BASE_VALIDATION_MAX_BATCHES = 128
GEOMETRY_NEIGHBOR_K = 8

# Experiment 2 - image identity one-vs-rest
RUN_IMAGE_IDENTITY_EXPERIMENT = False  # False = skip Experiment 2 entirely; set True once stimulus_presentations.image_name is available in the prepared .h5 files
# Set this to a real image name from the printed discovery-split image counts below when Experiment 2 is enabled.
IMAGE_TARGET_NAME = None
IMAGE_DISCOVERY_MAX_BATCHES = 128
IMAGE_DISCOVERY_TOP_K_CANDIDATES = 64
IMAGE_DISCOVERY_CANDIDATE_SESSION_BALANCE_FRACTION = 0.2
IMAGE_DISCOVERY_MIN_CANDIDATE_SCORE = 0.0
IMAGE_DISCOVERY_MIN_CLUSTER_SIZE = 2
IMAGE_VALIDATION_MAX_BATCHES = 128

# Experiment 3 - tighter stimulus_change discovery
TIGHT_DISCOVERY_MAX_BATCHES = 128
TIGHT_DISCOVERY_TOP_K_CANDIDATES = 32
TIGHT_DISCOVERY_CANDIDATE_SESSION_BALANCE_FRACTION = 0.2
TIGHT_DISCOVERY_MIN_CANDIDATE_SCORE = None
TIGHT_DISCOVERY_MIN_CLUSTER_SIZE = 3
TIGHT_VALIDATION_MAX_BATCHES = 128


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import json
import pandas as pd
from datetime import datetime

drive.mount('/content/drive')
repo_root = Path('/content/predictive_circuit_coding')
github_repo_url = 'https://github.com/jacobposchl/predictive_circuit_coding'
shared_drive_root = Path('/content/drive/MyDrive/pcc_runs')
drive_data_root = shared_drive_root / 'data' / 'allen_visual_behavior_neuropixels'
drive_export_root = Path('/content/drive/MyDrive/pcc_colab_outputs')
drive_export_root.mkdir(parents=True, exist_ok=True)

print('Repo root:', repo_root)
print('GitHub repo:', github_repo_url)
print('Shared dataset root:', drive_data_root)
print('Drive export root:', drive_export_root)

In [ ]:
if not repo_root.exists():
    !git clone {github_repo_url} {repo_root}
else:
    !git -C {repo_root} pull --ff-only
%cd {repo_root}
!git rev-parse --short HEAD
!pip install -e .

In [ ]:
repo_dataset_root = repo_root / 'data' / 'allen_visual_behavior_neuropixels'
repo_artifacts_root = repo_root / 'artifacts'

assert drive_data_root.exists(), f'Missing Drive dataset bundle: {drive_data_root}'

repo_dataset_root.parent.mkdir(parents=True, exist_ok=True)
if repo_dataset_root.exists() or repo_dataset_root.is_symlink():
    if repo_dataset_root.is_symlink():
        repo_dataset_root.unlink()
    else:
        shutil.rmtree(repo_dataset_root)
repo_dataset_root.symlink_to(drive_data_root, target_is_directory=True)

if not repo_artifacts_root.exists():
    repo_artifacts_root.mkdir(parents=True, exist_ok=True)

print('Linked dataset into repo:', repo_dataset_root)
print('Using local artifact directory:', repo_artifacts_root)

In [ ]:
import yaml
from predictive_circuit_coding.data import load_split_manifest
from predictive_circuit_coding.utils import (
    NotebookStageReporter,
    build_notebook_diagnostics_experiment_paths,
    build_notebook_diagnostics_export_path,
    build_notebook_diagnostics_local_root,
    build_notebook_discovery_runtime_config,
    build_notebook_diagnostics_summary_row,
    inspect_notebook_target_field_availability,
    describe_notebook_compute_targets,
    export_notebook_diagnostics_artifacts,
    find_existing_discovery_run,
    load_notebook_split_counts,
    materialize_notebook_prepared_sessions,
    resolve_notebook_checkpoint,
    restore_latest_exported_artifacts,
    run_notebook_discovery,
    run_notebook_geometry_diagnostics,
    run_notebook_validation,
    verify_paths_exist,
    write_notebook_diagnostics_summary,
)

reporter = NotebookStageReporter(name='colab-diagnostics', expected_duration='2-10 minutes for the full three-experiment sweep, longer for larger budgets')
reporter.banner('Representation And Motif Diagnostics', subtitle='Baseline geometry, image identity one-vs-rest, and tighter stimulus-change discovery')

data_config = repo_root / 'configs' / 'pcc' / 'allen_visual_behavior_neuropixels_local.yaml'
training_runtime_config = repo_root / 'colab_runtime_experiment.yaml'
checkpoint_dir = repo_root / 'artifacts' / 'checkpoints'
summary_path = repo_root / 'artifacts' / 'training_summary.json'

restored_training_export = restore_latest_exported_artifacts(
    drive_export_root=drive_export_root,
    local_artifact_root=repo_artifacts_root,
    runtime_experiment_config=training_runtime_config,
    training_run_id=TRAINING_RUN_ID,
)
if restored_training_export is None:
    raise FileNotFoundError('No exported training runs were found under Drive. Run the training notebook first so diagnostics can restore a selected run_id.')

selected_training_run_id = restored_training_export.parent.parent.name
diagnostics_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
local_diagnostics_root = build_notebook_diagnostics_local_root(local_artifact_root=repo_artifacts_root)
local_diagnostics_root.mkdir(parents=True, exist_ok=True)
drive_diagnostics_root = build_notebook_diagnostics_export_path(
    drive_export_root=drive_export_root,
    run_id=selected_training_run_id,
    diagnostics_timestamp=diagnostics_timestamp,
)

training_runtime_payload = yaml.safe_load(training_runtime_config.read_text(encoding='utf-8'))
runtime_subset_payload = training_runtime_payload.get('runtime_subset') or {}
split_manifest_path = Path(runtime_subset_payload.get('split_manifest_path', drive_data_root / 'splits' / 'split_manifest.json'))
split_counts = load_notebook_split_counts(split_manifest_path=split_manifest_path)

discovery_split_name = str((training_runtime_payload.get('splits') or {}).get('discovery', 'discovery'))
test_split_name = str((training_runtime_payload.get('splits') or {}).get('test', 'test'))
staged_dataset = None
if STAGE_PREPARED_SESSIONS_LOCALLY:
    split_manifest = load_split_manifest(split_manifest_path)
    session_ids_to_stage = sorted(
        {
            assignment.recording_id.split('/', 1)[1]
            for assignment in split_manifest.assignments
            if assignment.split in {discovery_split_name, test_split_name}
        }
    )
    staged_dataset = materialize_notebook_prepared_sessions(
        source_dataset_root=drive_data_root,
        target_dataset_root=repo_dataset_root,
        session_ids=session_ids_to_stage,
        dataset_id=str(training_runtime_payload.get('dataset_id', 'allen_visual_behavior_neuropixels')),
    )

selected_checkpoint = resolve_notebook_checkpoint(summary_path=summary_path, checkpoint_dir=checkpoint_dir)
image_target_field_debug = inspect_notebook_target_field_availability(
    experiment_config_path=training_runtime_config,
    data_config_path=data_config,
    split_name=discovery_split_name,
    target_label='stimulus_presentations.image_name',
)
image_value_counts = tuple(image_target_field_debug['value_counts'])

print('Requested training run ID:', TRAINING_RUN_ID or 'latest exported run')
print('Selected training run ID:', selected_training_run_id)
print('Restored training artifacts from:', restored_training_export)
print('Training runtime config:', training_runtime_config)
print('Resolved checkpoint:', selected_checkpoint)
print('Runtime split manifest:', split_manifest_path)
print('Realized split counts:', split_counts)
if staged_dataset is not None:
    print('Prepared-session staging: local copy enabled')
    print(f'Staged sessions locally: {len(staged_dataset.staged_session_ids)} | target={staged_dataset.target_prepared_root}')
else:
    print('Prepared-session staging: using Drive-mounted dataset tree')
print('Local diagnostics root:', local_diagnostics_root)
print('Drive diagnostics export root:', drive_diagnostics_root)
print('Diagnostics timestamp:', diagnostics_timestamp)
print('Top discovery-split image names (set IMAGE_TARGET_NAME from this list):')
display(pd.DataFrame(image_value_counts[:10]))
print('Image target field availability summary:')
print({
    'sessions_scanned': image_target_field_debug['sessions_scanned'],
    'sessions_with_stimulus_presentations': image_target_field_debug['sessions_with_namespace'],
    'sessions_with_image_name': image_target_field_debug['sessions_with_field'],
})
display(pd.DataFrame(image_target_field_debug['session_rows']))
image_experiment_enabled = bool(RUN_IMAGE_IDENTITY_EXPERIMENT)

In [ ]:
paths_ok = verify_paths_exist({
    "training_runtime_config": training_runtime_config,
    "data_config": data_config,
    "checkpoint": selected_checkpoint,
    "drive_dataset_root": drive_data_root,
})

print(paths_ok)

missing_paths = [name for name, exists in paths_ok.items() if not exists]
assert not missing_paths, f"Missing diagnostics notebook inputs: {', '.join(missing_paths)}"

available_image_names = {row["value"] for row in image_value_counts if row.get("value")}

print("Image identity experiment enabled:", image_experiment_enabled)

if image_experiment_enabled:
    assert IMAGE_TARGET_NAME is not None and str(IMAGE_TARGET_NAME).strip(), (
        "IMAGE_TARGET_NAME must be set when RUN_IMAGE_IDENTITY_EXPERIMENT = True."
    )

    assert available_image_names, (
        "No discovery-split image names were found. Inspect the image target field availability table above. "
        "If sessions_with_stimulus_presentations or sessions_with_image_name is 0, the staged prepared .h5 files "
        "do not expose stimulus_presentations.image_name, and the image-identity experiment cannot run until the "
        "dataset bundle is rebuilt/exported with that field."
    )

    assert IMAGE_TARGET_NAME in available_image_names, (
        f"IMAGE_TARGET_NAME '{IMAGE_TARGET_NAME}' was not found in the discovery split. "
        "Choose from the printed image counts above."
    )

    print("Using runtime subset described by the training notebook artifacts.")
    print("Image identity target:", IMAGE_TARGET_NAME)
else:
    print(
        "Skipping Experiment 2. Set RUN_IMAGE_IDENTITY_EXPERIMENT = True after the prepared "
        "dataset exposes stimulus_presentations.image_name."
    )
    print("Using runtime subset described by the training notebook artifacts.")

In [ ]:
reporter.begin('experiment 1', next_artifact='baseline discovery + validation + geometry summaries')
%cd {repo_root}

baseline_paths = build_notebook_diagnostics_experiment_paths(
    local_artifact_root=repo_artifacts_root,
    checkpoint_path=selected_checkpoint,
    experiment_name='baseline_stimulus_change',
)
baseline_runtime_config = build_notebook_discovery_runtime_config(
    source_experiment_config=training_runtime_config,
    runtime_experiment_config=baseline_paths.runtime_experiment_config_path,
    artifact_root=baseline_paths.experiment_root,
    decode_type=BASE_DECODE_TYPE,
    device_mode=DISCOVERY_DEVICE_MODE,
    step_log_every=NOTEBOOK_STEP_LOG_EVERY,
    discovery_max_batches=BASE_DISCOVERY_MAX_BATCHES,
    discovery_top_k_candidates=BASE_DISCOVERY_TOP_K_CANDIDATES,
    discovery_candidate_session_balance_fraction=BASE_DISCOVERY_CANDIDATE_SESSION_BALANCE_FRACTION,
    discovery_min_candidate_score=BASE_DISCOVERY_MIN_CANDIDATE_SCORE,
    discovery_min_cluster_size=BASE_DISCOVERY_MIN_CLUSTER_SIZE,
    validation_max_batches=BASE_VALIDATION_MAX_BATCHES,
)
baseline_compute_targets = describe_notebook_compute_targets(experiment_config_path=baseline_runtime_config)
print('Baseline runtime config:', baseline_runtime_config)
print('Baseline compute targets:', baseline_compute_targets)

baseline_discovery_run = find_existing_discovery_run(
    checkpoint_path=selected_checkpoint,
    split_name='discovery',
    target_label=BASE_DECODE_TYPE,
    experiment_config_path=baseline_runtime_config,
    discovery_output_path=baseline_paths.discovery_artifact_path,
)
baseline_discovery_reused = baseline_discovery_run is not None
if baseline_discovery_reused:
    print(f'Reusing existing baseline discovery run: {baseline_discovery_run.discovery_artifact_path}')
else:
    baseline_discovery_run = run_notebook_discovery(
        experiment_config_path=baseline_runtime_config,
        data_config_path=data_config,
        checkpoint_path=selected_checkpoint,
        split_name='discovery',
        output_path=baseline_paths.discovery_artifact_path,
        progress_ui=True,
    )

if baseline_discovery_reused and baseline_paths.validation_summary_json_path.exists() and baseline_paths.validation_summary_csv_path.exists():
    print(f'Reusing existing baseline validation run: {baseline_paths.validation_summary_json_path}')
    baseline_validation_run = type('BaselineValidationReuse', (), {
        'validation_summary_json_path': baseline_paths.validation_summary_json_path,
        'validation_summary_csv_path': baseline_paths.validation_summary_csv_path,
    })()
else:
    baseline_validation_run = run_notebook_validation(
        experiment_config_path=baseline_runtime_config,
        data_config_path=data_config,
        checkpoint_path=selected_checkpoint,
        discovery_artifact_path=baseline_discovery_run.discovery_artifact_path,
        output_json_path=baseline_paths.validation_summary_json_path,
        output_csv_path=baseline_paths.validation_summary_csv_path,
        progress_ui=True,
    )

if baseline_discovery_reused and baseline_paths.window_geometry_summary_json_path.exists() and baseline_paths.candidate_geometry_summary_json_path.exists():
    print(f'Reusing existing baseline geometry summaries: {baseline_paths.window_geometry_summary_json_path}')
    baseline_geometry_run = type('BaselineGeometryReuse', (), {
        'window_geometry_summary_json_path': baseline_paths.window_geometry_summary_json_path,
        'window_geometry_summary_csv_path': baseline_paths.window_geometry_summary_csv_path,
        'candidate_geometry_summary_json_path': baseline_paths.candidate_geometry_summary_json_path,
        'candidate_geometry_summary_csv_path': baseline_paths.candidate_geometry_summary_csv_path,
    })()
else:
    baseline_geometry_run = run_notebook_geometry_diagnostics(
        experiment_config_path=baseline_runtime_config,
        data_config_path=data_config,
        checkpoint_path=selected_checkpoint,
        discovery_artifact_path=baseline_discovery_run.discovery_artifact_path,
        neighbor_k=GEOMETRY_NEIGHBOR_K,
        output_json_path=baseline_paths.window_geometry_summary_json_path,
        output_csv_path=baseline_paths.window_geometry_summary_csv_path,
        candidate_output_json_path=baseline_paths.candidate_geometry_summary_json_path,
        candidate_output_csv_path=baseline_paths.candidate_geometry_summary_csv_path,
        progress_ui=True,
    )

print('Baseline discovery artifact:', baseline_discovery_run.discovery_artifact_path)
print('Baseline validation summary:', baseline_validation_run.validation_summary_json_path)
print('Baseline window geometry:', baseline_geometry_run.window_geometry_summary_json_path)
print('Baseline candidate geometry:', baseline_geometry_run.candidate_geometry_summary_json_path)
reporter.finish('experiment 1')

In [ ]:
reporter.begin('experiment 2', next_artifact='image-identity one-vs-rest discovery + validation')
%cd {repo_root}

image_experiment_name = None
image_paths = None
image_discovery_run = None
image_validation_run = None
if image_experiment_enabled:
    image_experiment_name = f"image_identity_{IMAGE_TARGET_NAME}".replace('/', '_').replace(' ', '_')
    image_paths = build_notebook_diagnostics_experiment_paths(
        local_artifact_root=repo_artifacts_root,
        checkpoint_path=selected_checkpoint,
        experiment_name=image_experiment_name,
    )
    image_runtime_config = build_notebook_discovery_runtime_config(
        source_experiment_config=training_runtime_config,
        runtime_experiment_config=image_paths.runtime_experiment_config_path,
        artifact_root=image_paths.experiment_root,
        decode_type='stimulus_presentations.image_name',
        target_label_mode='onset_within_window',
        target_label_match_value=IMAGE_TARGET_NAME,
        device_mode=DISCOVERY_DEVICE_MODE,
        step_log_every=NOTEBOOK_STEP_LOG_EVERY,
        discovery_max_batches=IMAGE_DISCOVERY_MAX_BATCHES,
        discovery_top_k_candidates=IMAGE_DISCOVERY_TOP_K_CANDIDATES,
        discovery_candidate_session_balance_fraction=IMAGE_DISCOVERY_CANDIDATE_SESSION_BALANCE_FRACTION,
        discovery_min_candidate_score=IMAGE_DISCOVERY_MIN_CANDIDATE_SCORE,
        discovery_min_cluster_size=IMAGE_DISCOVERY_MIN_CLUSTER_SIZE,
        validation_max_batches=IMAGE_VALIDATION_MAX_BATCHES,
    )
    print('Image runtime config:', image_runtime_config)

    image_discovery_run = find_existing_discovery_run(
        checkpoint_path=selected_checkpoint,
        split_name='discovery',
        target_label='stimulus_presentations.image_name',
        experiment_config_path=image_runtime_config,
        discovery_output_path=image_paths.discovery_artifact_path,
    )
    image_discovery_reused = image_discovery_run is not None
    if image_discovery_reused:
        print(f'Reusing existing image discovery run: {image_discovery_run.discovery_artifact_path}')
    else:
        image_discovery_run = run_notebook_discovery(
            experiment_config_path=image_runtime_config,
            data_config_path=data_config,
            checkpoint_path=selected_checkpoint,
            split_name='discovery',
            output_path=image_paths.discovery_artifact_path,
            progress_ui=True,
        )

    if image_discovery_reused and image_paths.validation_summary_json_path.exists() and image_paths.validation_summary_csv_path.exists():
        print(f'Reusing existing image validation run: {image_paths.validation_summary_json_path}')
        image_validation_run = type('ImageValidationReuse', (), {
            'validation_summary_json_path': image_paths.validation_summary_json_path,
            'validation_summary_csv_path': image_paths.validation_summary_csv_path,
        })()
    else:
        image_validation_run = run_notebook_validation(
            experiment_config_path=image_runtime_config,
            data_config_path=data_config,
            checkpoint_path=selected_checkpoint,
            discovery_artifact_path=image_discovery_run.discovery_artifact_path,
            output_json_path=image_paths.validation_summary_json_path,
            output_csv_path=image_paths.validation_summary_csv_path,
            progress_ui=True,
        )

    print('Image discovery artifact:', image_discovery_run.discovery_artifact_path)
    print('Image validation summary:', image_validation_run.validation_summary_json_path)
else:
    print('Experiment 2 skipped.')
reporter.finish('experiment 2')

In [ ]:
reporter.begin('experiment 3', next_artifact='tighter stimulus-change discovery + validation')
%cd {repo_root}

tight_paths = build_notebook_diagnostics_experiment_paths(
    local_artifact_root=repo_artifacts_root,
    checkpoint_path=selected_checkpoint,
    experiment_name='tight_stimulus_change',
)
tight_runtime_config = build_notebook_discovery_runtime_config(
    source_experiment_config=training_runtime_config,
    runtime_experiment_config=tight_paths.runtime_experiment_config_path,
    artifact_root=tight_paths.experiment_root,
    decode_type=BASE_DECODE_TYPE,
    device_mode=DISCOVERY_DEVICE_MODE,
    step_log_every=NOTEBOOK_STEP_LOG_EVERY,
    discovery_max_batches=TIGHT_DISCOVERY_MAX_BATCHES,
    discovery_top_k_candidates=TIGHT_DISCOVERY_TOP_K_CANDIDATES,
    discovery_candidate_session_balance_fraction=TIGHT_DISCOVERY_CANDIDATE_SESSION_BALANCE_FRACTION,
    discovery_min_candidate_score=TIGHT_DISCOVERY_MIN_CANDIDATE_SCORE,
    discovery_min_cluster_size=TIGHT_DISCOVERY_MIN_CLUSTER_SIZE,
    validation_max_batches=TIGHT_VALIDATION_MAX_BATCHES,
)
print('Tight runtime config:', tight_runtime_config)

tight_discovery_run = find_existing_discovery_run(
    checkpoint_path=selected_checkpoint,
    split_name='discovery',
    target_label=BASE_DECODE_TYPE,
    experiment_config_path=tight_runtime_config,
    discovery_output_path=tight_paths.discovery_artifact_path,
)
tight_discovery_reused = tight_discovery_run is not None
if tight_discovery_reused:
    print(f'Reusing existing tight discovery run: {tight_discovery_run.discovery_artifact_path}')
else:
    tight_discovery_run = run_notebook_discovery(
        experiment_config_path=tight_runtime_config,
        data_config_path=data_config,
        checkpoint_path=selected_checkpoint,
        split_name='discovery',
        output_path=tight_paths.discovery_artifact_path,
        progress_ui=True,
    )

if tight_discovery_reused and tight_paths.validation_summary_json_path.exists() and tight_paths.validation_summary_csv_path.exists():
    print(f'Reusing existing tight validation run: {tight_paths.validation_summary_json_path}')
    tight_validation_run = type('TightValidationReuse', (), {
        'validation_summary_json_path': tight_paths.validation_summary_json_path,
        'validation_summary_csv_path': tight_paths.validation_summary_csv_path,
    })()
else:
    tight_validation_run = run_notebook_validation(
        experiment_config_path=tight_runtime_config,
        data_config_path=data_config,
        checkpoint_path=selected_checkpoint,
        discovery_artifact_path=tight_discovery_run.discovery_artifact_path,
        output_json_path=tight_paths.validation_summary_json_path,
        output_csv_path=tight_paths.validation_summary_csv_path,
        progress_ui=True,
    )

print('Tight discovery artifact:', tight_discovery_run.discovery_artifact_path)
print('Tight validation summary:', tight_validation_run.validation_summary_json_path)
reporter.finish('experiment 3')

In [ ]:
reporter.begin('summary', next_artifact='combined diagnostics summary json/csv + Drive export')

summary_rows = [
    build_notebook_diagnostics_summary_row(
        experiment_name='baseline_stimulus_change',
        discovery_artifact_path=baseline_paths.discovery_artifact_path,
        validation_summary_path=baseline_paths.validation_summary_json_path,
        cluster_summary_path=baseline_paths.cluster_summary_json_path,
        window_geometry_summary_path=baseline_paths.window_geometry_summary_json_path,
        candidate_geometry_summary_path=baseline_paths.candidate_geometry_summary_json_path,
    ),
    build_notebook_diagnostics_summary_row(
        experiment_name='tight_stimulus_change',
        discovery_artifact_path=tight_paths.discovery_artifact_path,
        validation_summary_path=tight_paths.validation_summary_json_path,
        cluster_summary_path=tight_paths.cluster_summary_json_path,
    ),
]
if image_experiment_enabled and image_paths is not None:
    summary_rows.insert(
        1,
        build_notebook_diagnostics_summary_row(
            experiment_name=image_experiment_name,
            discovery_artifact_path=image_paths.discovery_artifact_path,
            validation_summary_path=image_paths.validation_summary_json_path,
            cluster_summary_path=image_paths.cluster_summary_json_path,
        ),
    )
summary_json_path, summary_csv_path = write_notebook_diagnostics_summary(
    summary_rows,
    output_json_path=local_diagnostics_root / 'combined_experiment_summary.json',
    output_csv_path=local_diagnostics_root / 'combined_experiment_summary.csv',
)
summary_df = pd.DataFrame(summary_rows)

for row in summary_rows:
    print(
        f"{row['experiment_name']}: target={row['target_label']} match={row['target_label_match_value']} | "
        f"clusters={row['cluster_count']} candidates={row['candidate_count']} | "
        f"real={row['real_probe_accuracy']:.3f} shuffled={row['shuffled_probe_accuracy']:.3f} | "
        f"held_out_acc={row['held_out_test_probe_accuracy']:.3f} roc_auc={row['held_out_similarity_roc_auc']:.3f} pr_auc={row['held_out_similarity_pr_auc']:.3f}"
    )

display(summary_df)
drive_diagnostics_export_path = export_notebook_diagnostics_artifacts(
    drive_export_root=drive_export_root,
    local_artifact_root=repo_artifacts_root,
    run_id=selected_training_run_id,
    diagnostics_timestamp=diagnostics_timestamp,
)
print('Combined summary JSON:', summary_json_path)
print('Combined summary CSV:', summary_csv_path)
print('Diagnostics artifacts saved to Drive:', drive_diagnostics_export_path)
print('Baseline experiment root:', baseline_paths.experiment_root)
if image_experiment_enabled and image_paths is not None:
    print('Image experiment root:', image_paths.experiment_root)
else:
    print('Image experiment root: skipped')
print('Tight experiment root:', tight_paths.experiment_root)
reporter.finish('summary')